In [ ]:
from utils.postgres_util import get_engine_from_env
from utils.producer_queue_manager import (
    ensure_send_queue_runtime_columns,
    ensure_simulation_state_control_table,
    upsert_simulation_state_control,
    read_simulation_state_control,
    claim_pending_send_queue_batch,
    mark_claimed_batch_sent, 
    mark_claimed_batch_failed,
    requeue_failed_messages,
    release_stale_claims,
    get_send_queue_status_counts
)



In [ ]:

engine = get_engine_from_env()


In [ ]:

ensure_send_queue_runtime_columns(
    engine=engine,
    schema="capstone",
    table_name="synthetic_sensor_messages_send_queue",
)


In [ ]:

ensure_simulation_state_control_table(
    engine=engine,
    schema="capstone",
    table_name="simulation_state_control",
)


In [ ]:

upsert_simulation_state_control(
    engine=engine,
    dataset_id="pump_synthetic_v1",
    run_id="premelt_run_001",
    is_enabled=True,
    producer_topic="pump.telemetry.synthetic",
    producer_batch_size=500,
    producer_poll_seconds=0.0,
    max_send_attempts=3,
    schema="capstone",
    table_name="simulation_state_control",
)


In [ ]:

control_row = read_simulation_state_control(
    engine=engine,
    dataset_id="pump_synthetic_v1",
    run_id="premelt_run_001",
    schema="capstone",
    table_name="simulation_state_control",
)


In [ ]:

control_row

In [ ]:
claimed_dataframe = claim_pending_send_queue_batch(
    engine=engine,
    batch_size=500,
    schema="capstone",
    table_name="synthetic_sensor_messages_send_queue",
    producer_topic="pump.telemetry.synthetic",
    producer_worker_id="producer_worker_001",
)

print("Claimed rows:", len(claimed_dataframe))



In [ ]:
display(
    claimed_dataframe[
        [
            "claim_token",
            "observation_index",
            "message_sequence_index",
            "sensor_name",
            "sensor_index",
            "queue_status",
            "producer_topic",
            "producer_worker_id",
        ]
    ].head(20)
)

In [ ]:
if not claimed_dataframe.empty:
    claim_token = str(claimed_dataframe["claim_token"].iloc[0])

    sent_dataframe = mark_claimed_batch_sent(
        engine=engine,
        claim_token=claim_token,
        schema="capstone",
        table_name="synthetic_sensor_messages_send_queue",
    )

    print("Marked sent:", len(sent_dataframe))
    display(sent_dataframe)
else:
    print("No claimed rows to mark as sent.")

In [ ]:
if not claimed_dataframe.empty:
    claim_token = str(claimed_dataframe["claim_token"].iloc[0])

    failed_dataframe = mark_claimed_batch_failed(
        engine=engine,
        claim_token=claim_token,
        error_message="Kafka publish failed during test run.",
        schema="capstone",
        table_name="synthetic_sensor_messages_send_queue",
    )

    print("Marked failed:", len(failed_dataframe))
    display(failed_dataframe.head())
else:
    print("No claimed rows to mark as failed.")

In [ ]:
requeued_dataframe = requeue_failed_messages(
    engine=engine,
    max_send_attempts=3,
    schema="capstone",
    table_name="synthetic_sensor_messages_send_queue",
)

print("Requeued failed rows:", len(requeued_dataframe))
display(requeued_dataframe.head())

In [ ]:
released_dataframe = release_stale_claims(
    engine=engine,
    stale_after_minutes=15,
    max_send_attempts=3,
    schema="capstone",
    table_name="synthetic_sensor_messages_send_queue",
)

print("Released stale claims:", len(released_dataframe))
display(released_dataframe.head())

In [ ]:
status_dataframe = get_send_queue_status_counts(
    engine=engine,
    schema="public",
    table_name="synthetic_sensor_messages_send_queue",
)

display(status_dataframe)